In [1]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
import datetime as dt
from pathlib import Path
import os
from tqdm import tqdm

In [2]:
# Find nearest index
def find_index(array, x):
    if array.ndim == 1:
        idx = np.argmin(np.abs(array - x))
    elif array.ndim == 2:
        idx = np.unravel_index(np.argmin(np.abs(array - x)), array.shape)
    else:
        raise ValueError("Unsupported array dimensions for find_index function.")
    return idx

In [ ]:
soa_result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_SOAvars/')
daily_soa_to_open = 'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.*SFWET.nc'

nc_daily_soa = xr.open_mfdataset(str(soa_result_dir/daily_soa_to_open),combine='nested')
#nc_daily_soa

#Read in pom wet dep timeseries files
pom_result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_h2/')
daily_pom_to_open = 'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.pom_*SFWET.nc'

nc_daily_pom = xr.open_mfdataset(str(pom_result_dir/daily_pom_to_open),combine='nested')
#nc_daily_pom

#merge SOA and POM files together
nc_daily_oc = xr.merge([nc_daily_soa, nc_daily_pom])
nc_daily_oc 

In [5]:
#################### WET DEP Carbon ############################################3
#DOC wet = pom, soa (SFWET)
nc_daily_oc['DOC_wet_sum'] = (nc_daily_oc['soa1_a1SFWET'] + nc_daily_oc['soa1_a2SFWET'] + nc_daily_oc['soa1_c1SFWET'] 
                             + nc_daily_oc['soa1_c2SFWET'] + nc_daily_oc['soa2_a1SFWET'] + nc_daily_oc['soa2_a2SFWET'] 
                             + nc_daily_oc['soa2_c1SFWET'] +  nc_daily_oc['soa2_c2SFWET'] + nc_daily_oc['soa3_a1SFWET'] 
                             + nc_daily_oc['soa3_a2SFWET'] + nc_daily_oc['soa3_c1SFWET'] + nc_daily_oc['soa3_c2SFWET'] 
                             + nc_daily_oc['soa4_a1SFWET'] + nc_daily_oc['soa4_a2SFWET'] + nc_daily_oc['soa4_c1SFWET'] 
                             + nc_daily_oc['soa4_c2SFWET'] + nc_daily_oc['soa5_a1SFWET'] + nc_daily_oc['soa5_a2SFWET'] 
                             + nc_daily_oc['soa5_c1SFWET'] + nc_daily_oc['soa5_c2SFWET'] + nc_daily_oc['pom_a1SFWET']
                             + nc_daily_oc['pom_a4SFWET'] + nc_daily_oc['pom_c1SFWET'] + nc_daily_oc['pom_c4SFWET']) * -1

# Perform unit conversions
nc_daily_oc['DOC_wet_mgm2'] = nc_daily_oc['DOC_wet_sum'] * (86400 * 1000000)
nc_daily_doc = nc_daily_oc[['DOC_wet_mgm2']]
nc_daily_doc

<xarray.Dataset>
Dimensions:       (time: 7671, lat: 192, lon: 288)
Coordinates:
  * lat           (lat) float64 -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon           (lon) float64 0.0 1.25 2.5 3.75 ... 355.0 356.2 357.5 358.8
  * time          (time) datetime64[ns] 2002-01-01 2002-01-02 ... 2023-01-01
Data variables:
    DOC_wet_mgm2  (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Apr11_01.2002_2023.001
    logname:           demurray
    host:              derecho1
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [6]:
#Select cells that correspond to NADP sites: read in NADP lat/long and apply the find nearest function
pathData = '/glade/u/home/demurray/External File Uploads'
os.chdir(pathData)
ntn = pd.read_csv('ntn.csv')

# Only select the cells in .nc that correspond to an NADP site lat/long
subset_list = []
progress_bar = tqdm(total=len(ntn))
for index, row in ntn.iterrows():
    lat = row['latitude']
    lon = 360-(row['longitude']*-1)   # longitude is positive and based on 360 degrees.
    lat_idx = find_index(nc_daily_oc['lat'].values, lat)   # right now we are doing a 'find nearest' calculation, should probably interpolate across grid cell and have exact coordinates represented?
    lon_idx = find_index(nc_daily_oc['lon'].values, lon)
    subset = nc_daily_oc.isel(lat=lat_idx, lon=lon_idx)
    subset['siteId'] = row['siteId']  # Add 'siteId' as a new coordinate/index
    subset = subset.assign_coords(siteId=row['siteId'])
    subset_list.append(subset)
    progress_bar.update(1)
progress_bar.close()

# Concatenate the list of subsets into a new xarray dataset
nc_daily_nadp = xr.concat(subset_list, dim='siteId')
nc_daily_nadp

100%|██████████| 391/391 [00:04<00:00, 97.27it/s] 


<xarray.Dataset>
Dimensions:       (siteId: 391, time: 7671)
Coordinates:
    lat           (siteId) float64 57.02 56.07 57.02 65.5 ... 42.88 39.11 40.99
    lon           (siteId) float64 248.8 248.8 248.8 212.5 ... 271.2 280.0 253.8
  * time          (time) datetime64[ns] 2002-01-01 2002-01-02 ... 2023-01-01
  * siteId        (siteId) <U4 'AB32' 'AB34' 'AB36' ... 'WI99' 'WV99' 'WY96'
Data variables: (12/26)
    soa1_a1SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    soa1_a2SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    soa1_c1SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    soa1_c2SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    soa2_a1SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    soa2_a2SFWET  (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    ...            ...
    pom_a1SFWET   (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    pom_a4SFWET   (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    pom_c1SFWET   (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    pom_c4SFWET   (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    DOC_wet_sum   (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    DOC_wet_mgm2  (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Apr11_01.2002_2023.001
    logname:           demurray
    host:              derecho1
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [ ]:
#NEED TO RUN THIS CODE STILL FOR WRITING DAILY DOC TIMESERIES FILES
#Add a loop that for each siteId it writes a file to timeseries
output_directory = '/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites'

unique_site_ids = nc_daily_nadp['siteId'].values

# Initialize tqdm
pbar = tqdm(unique_site_ids, desc="Writing subset files")

# Iterate over each unique siteId
for site_id in pbar:
    # Subset the dataset for the current siteId
    subset_ds = nc_daily_nadp.where(nc_daily_nadp['siteId'] == site_id, drop=True)
    
    # Construct the filename
    filename = f'{site_id}_DailyTimeseries_WetDep_DOC.nc'
    
    # Write the subsetted dataset to the specified directory
    output_path = os.path.join(output_directory, filename)
    subset_ds.to_netcdf(output_path)
    
    # Update tqdm description
    pbar.set_description(f"Writing subset files: {filename}")

Writing subset files: MI26_DailyTimeseries_WetDep_DOC.nc:  28%|██▊       | 111/391 [22:23:22<126:43:04, 1629.23s/it]

In [8]:
#Subset by time stamps we have DOC wet dep data for (2017 - 2018)

# Define the time range in the same format as the datetime64 variable
start_date = '2017-01-01T00:00:00.000000000'
end_date = '2023-01-01T23:59:59.999999999'

# Subset the dataset based on time
nc_daily_subset = nc_daily_nadp.sel(time=slice(start_date, end_date))
nc_daily_subset

<xarray.Dataset>
Dimensions:       (siteId: 391, time: 731)
Coordinates:
    lat           (siteId) float64 57.02 56.07 57.02 65.5 ... 42.88 39.11 40.99
    lon           (siteId) float64 248.8 248.8 248.8 212.5 ... 271.2 280.0 253.8
  * time          (time) datetime64[ns] 2017-01-01 2017-01-02 ... 2019-01-01
  * siteId        (siteId) <U4 'AB32' 'AB34' 'AB36' ... 'WI99' 'WV99' 'WY96'
Data variables:
    DOC_wet_mgm2  (siteId, time) float64 dask.array<chunksize=(1, 731), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Apr11_01.2002_2023.001
    logname:           demurray
    host:              derecho1
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [10]:
#Turn nc_daily_nadp xarray into a pandas dataframe with similar attributes to the NADP dataset
mod_nadp = nc_daily_subset.to_dataframe().reset_index()
mod_nadp = mod_nadp.rename(columns = {'time': 'model_time', 'DOC_wet_mgm2': 'Mod_DOC_mgm2'})

#IMPORTANT STEP: change the time to one day prior (model writes time at end of current day)
mod_nadp['time'] = pd.to_datetime(mod_nadp['model_time'],format='%Y-%m-%d')+DateOffset(days=-1)

mod_nadp.head(10)

,siteId,model_time,Mod_DOC_mgm2,lat,lon,time
0,AB32,2017-01-01,4.440780e-01,57.015707,248.75,2016-12-31
1,AB32,2017-01-02,7.728940e-02,57.015707,248.75,2017-01-01
2,AB32,2017-01-03,1.311704e-04,57.015707,248.75,2017-01-02
3,AB32,2017-01-04,9.553314e-03,57.015707,248.75,2017-01-03
4,AB32,2017-01-05,6.175148e-03,57.015707,248.75,2017-01-04
5,AB32,2017-01-06,1.120694e-02,57.015707,248.75,2017-01-05
6,AB32,2017-01-07,9.965356e-07,57.015707,248.75,2017-01-06
7,AB32,2017-01-08,3.961280e-03,57.015707,248.75,2017-01-07
8,AB32,2017-01-09,3.018631e-04,57.015707,248.75,2017-01-08
9,AB32,2017-01-10,1.022313e-02,57.015707,248.75,2017-01-09


In [14]:
#read in timeseries of: NADP NTN and ensure correct/consistent formatting
pathData = '/glade/u/home/demurray/External File Uploads'
os.chdir(pathData)
nadp_df = pd.read_csv('NADP DOM data clean.csv', parse_dates = ['dateOn', 'dateOff'])
nadp_df.head()

#Scale concentrations to mg/m2 using precip volume, assuming 1mm rain = 1L/m2
nadp_df['DOC_mgm2'] = (nadp_df['DOC_mgL_Final_UNH'] * nadp_df['subppt']) 

#Ensure correct datetime formatting
nadp_df['dateOn'] = pd.to_datetime(nadp_df['dateOn'], format='%m/%d/%Y %H:%M')
nadp_df['dateOff'] = pd.to_datetime(nadp_df['dateOff'], format='%m/%d/%Y %H:%M')
nadp_df = nadp_df.sort_values(['siteId', 'dateOn'], ascending = True)

#Need to round date because using DAILY data for modelled comparisons
nadp_df['dateOnround'] = nadp_df.dateOn + dt.timedelta(hours=12)
nadp_df['dateOnround'] = pd.to_datetime(nadp_df.dateOnround.dt.strftime('%Y-%m-%d'))
nadp_df['dateOffround'] = nadp_df.dateOff + dt.timedelta(hours=12)
nadp_df['dateOffround'] = pd.to_datetime(nadp_df.dateOffround.dt.strftime('%Y-%m-%d'))

#merge with site info and then select relevant columns
nadp_df = pd.merge(nadp_df, ntn, on = 'siteId')
nadp_df = nadp_df[['siteId', 'latitude', 'longitude', 'dateOn', 'dateOff', 'dateOnround', 'dateOffround', 'subppt', 'DOC_mgm2']] 
nadp_df.head(5)

,siteId,latitude,longitude,dateOn,dateOff,dateOnround,dateOffround,subppt,DOC_mgm2
0,AK02,58.5139,-134.7843,2017-02-21 18:05:00,2017-02-28 18:28:00,2017-02-22,2017-03-01,8.128,3.16992
1,AK02,58.5139,-134.7843,2017-03-21 18:55:00,2017-03-28 17:50:00,2017-03-22,2017-03-29,12.192,5.97408
2,AK02,58.5139,-134.7843,2017-04-18 18:07:00,2017-04-25 18:01:00,2017-04-19,2017-04-26,8.128,4.79552
3,AK02,58.5139,-134.7843,2017-05-16 16:18:00,2017-05-23 18:23:00,2017-05-17,2017-05-24,39.116,18.77568
4,AK02,58.5139,-134.7843,2017-06-13 18:02:00,2017-06-20 17:17:00,2017-06-14,2017-06-21,78.994,33.96742


In [15]:
##Assign sampling intervals to NADP NTN deposition data
nadp_df['SamplingInt'] = pd.Series(dtype='int')
nadp_df['IntTime'] = pd.Series(dtype='int')

sites = nadp_df.siteId.unique()

for i in tqdm(sites, unit = 'sites', total = len(sites), ncols = 100):
    nadp_df.loc[nadp_df.siteId == i,'SamplingInt'] = list(range(0, len(nadp_df.loc[nadp_df.siteId == i]), 1))
    nadp_df.loc[nadp_df.siteId == i, 'IntTime'] = nadp_df.loc[nadp_df.siteId == i, 'dateOff'] - nadp_df.loc[nadp_df.siteId == i, 'dateOn']
nadp_df.head(10)

100%|███████████████████████████████████████████████████████████| 17/17 [00:00<00:00, 223.57sites/s]


,siteId,latitude,longitude,dateOn,dateOff,dateOnround,dateOffround,subppt,DOC_mgm2,SamplingInt,IntTime
0,AK02,58.5139,-134.7843,2017-02-21 18:05:00,2017-02-28 18:28:00,2017-02-22,2017-03-01,8.128,3.16992,0.0,7 days 00:23:00
1,AK02,58.5139,-134.7843,2017-03-21 18:55:00,2017-03-28 17:50:00,2017-03-22,2017-03-29,12.192,5.97408,1.0,6 days 22:55:00
2,AK02,58.5139,-134.7843,2017-04-18 18:07:00,2017-04-25 18:01:00,2017-04-19,2017-04-26,8.128,4.79552,2.0,6 days 23:54:00
3,AK02,58.5139,-134.7843,2017-05-16 16:18:00,2017-05-23 18:23:00,2017-05-17,2017-05-24,39.116,18.77568,3.0,7 days 02:05:00
4,AK02,58.5139,-134.7843,2017-06-13 18:02:00,2017-06-20 17:17:00,2017-06-14,2017-06-21,78.994,33.96742,4.0,6 days 23:15:00
5,AK02,58.5139,-134.7843,2017-07-11 17:25:00,2017-07-18 18:11:00,2017-07-12,2017-07-19,77.470,32.53740,5.0,7 days 00:46:00
6,AK02,58.5139,-134.7843,2017-08-08 17:11:00,2017-08-15 22:30:00,2017-08-09,2017-08-16,47.244,11.33856,6.0,7 days 05:19:00
7,AK02,58.5139,-134.7843,2017-09-12 18:20:00,2017-09-19 18:50:00,2017-09-13,2017-09-20,11.430,4.34340,7.0,7 days 00:30:00
8,AK02,58.5139,-134.7843,2017-10-03 19:12:00,2017-10-10 18:50:00,2017-10-04,2017-10-11,53.848,11.30808,8.0,6 days 23:38:00
9,AK02,58.5139,-134.7843,2017-11-21 19:35:00,2017-11-28 19:41:00,2017-11-22,2017-11-29,30.734,13.52296,9.0,7 days 00:06:00


In [18]:
#Write loop to assign sampling intervals to  modelled data frame
mod_nadp['SamplingInt'] = pd.Series(dtype='int') # Add a SamplingInt column to the modelled
sites =nadp_df.siteId.unique() 

for i in tqdm(sites, unit = "sites", total = len(sites), ncols = 100):
    sampleInt = nadp_df.loc[nadp_df.siteId==i,'SamplingInt']
    #print(i)
    for j in sampleInt:
       #print(j)
       begDate = pd.Timestamp(nadp_df.loc[(nadp_df.siteId == i) & (nadp_df.SamplingInt == j), 'dateOn'].item())
       endDate = pd.Timestamp(nadp_df.loc[(nadp_df.siteId == i) & (nadp_df.SamplingInt == j), 'dateOff'].item())
       #print(endDate)
       mod_nadp.loc[(mod_nadp.siteId == i) & (mod_nadp.time >= begDate) & (mod_nadp.time < endDate), 'SamplingInt'] = j 

mod_nadp.drop_duplicates(inplace = True)
mod_nadp.dropna(subset = ['SamplingInt'], inplace = True)
mod_nadp.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites/DOC_wetdep_Daily_SamplingIntAssigned.csv')
mod_nadp.head(20)

100%|████████████████████████████████████████████████████████████| 17/17 [00:05<00:00,  2.98sites/s]


,siteId,model_time,Mod_DOC_mgm2,lat,lon,time,SamplingInt
2977,AK02,2017-02-23,4.087467e-03,58.900524,225.0,2017-02-22,0.0
2978,AK02,2017-02-24,3.568007e-02,58.900524,225.0,2017-02-23,0.0
2979,AK02,2017-02-25,6.359636e-04,58.900524,225.0,2017-02-24,0.0
2980,AK02,2017-02-26,1.967692e-01,58.900524,225.0,2017-02-25,0.0
2981,AK02,2017-02-27,9.516438e-03,58.900524,225.0,2017-02-26,0.0
2982,AK02,2017-02-28,1.094379e-02,58.900524,225.0,2017-02-27,0.0
2983,AK02,2017-03-01,7.814593e-02,58.900524,225.0,2017-02-28,0.0
3005,AK02,2017-03-23,4.541033e-02,58.900524,225.0,2017-03-22,1.0
3006,AK02,2017-03-24,1.551255e-01,58.900524,225.0,2017-03-23,1.0
3007,AK02,2017-03-25,2.100217e-02,58.900524,225.0,2017-03-24,1.0


In [19]:
#Read in the new precip files
precip_result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_h2/')
precip_files_to_open = ['FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.PRECC.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.PRECL.nc']

# Concatenate directory path with each file name separately
file_paths = [precip_result_dir / file_name for file_name in precip_files_to_open]

# Open multiple netCDF files as a single dataset
nc_precip = xr.open_mfdataset(file_paths, combine='nested') # units are in m/s
nc_precip['PRECC_mm'] = nc_precip['PRECC']* (86400*1000) # seconds to days and m to mm = mm/d
nc_precip['PRECL_mm'] = nc_precip['PRECL']* (86400*1000)
nc_precip['PREC_tot_mm'] = nc_precip['PRECC_mm']  + nc_precip['PRECL_mm']

#Subset for date range
nc_precip = nc_precip.sel(time=slice(start_date, end_date))

# Only select the cells in .nc that correspond to an NADP site lat/long
subset_list = []
progress_bar = tqdm(total=len(ntn))
for index, row in ntn.iterrows():
    lat = row['latitude']
    lon = 360-(row['longitude']*-1)   # longitude is positive and based on 360 degrees.
    lat_idx = find_index(nc_precip['lat'].values, lat)   # right now we are doing a 'find nearest' calculation, should probably interpolate across grid cell and have exact coordinates represented?
    lon_idx = find_index(nc_precip['lon'].values, lon)
    subset = nc_precip.isel(lat=lat_idx, lon=lon_idx)
    subset['siteId'] = row['siteId']  # Add 'siteId' as a new coordinate/index
    subset = subset.assign_coords(siteId=row['siteId'])
    subset_list.append(subset)
    progress_bar.update(1)
progress_bar.close()

# Concatenate the list of subsets into a new xarray dataset
nc_precip_nadp = xr.concat(subset_list, dim='siteId')

#Convert to dataframe and assign new time
mod_nadp_precip = nc_precip_nadp.to_dataframe().reset_index()
mod_nadp_precip = mod_nadp_precip.rename(columns = {'time': 'model_time'})

#IMPORTANT STEP: change the time to one day prior (model writes time at end of current day)
mod_nadp_precip['time'] = pd.to_datetime(mod_nadp_precip['model_time'],format='%Y-%m-%d')+DateOffset(days=-1)
mod_nadp_precip = mod_nadp_precip[['siteId', 'time', 'PREC_tot_mm', 'PRECC_mm', 'PRECL_mm']]
mod_nadp_precip.head(10)

100%|██████████| 391/391 [00:00<00:00, 450.92it/s]


,siteId,time,PREC_tot_mm,PRECC_mm,PRECL_mm
0,AB32,2016-12-31,1.612292,0.0,1.612292
1,AB32,2017-01-01,0.341198,0.0,0.341198
2,AB32,2017-01-02,0.000230,0.0,0.000230
3,AB32,2017-01-03,0.284618,0.0,0.284618
4,AB32,2017-01-04,0.736271,0.0,0.736271
5,AB32,2017-01-05,0.470006,0.0,0.470006
6,AB32,2017-01-06,0.009303,0.0,0.009303
7,AB32,2017-01-07,0.165363,0.0,0.165363
8,AB32,2017-01-08,0.100278,0.0,0.100278
9,AB32,2017-01-09,0.109422,0.0,0.109422


In [20]:
#Merge with precip df on siteId and time
mod_nadp_all = pd.merge(mod_nadp, mod_nadp_precip, on = ['siteId', 'time'])
mod_nadp_all.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites/PREC_DOC_wetdep_Daily_SamplingIntAssigned_Allsites.csv')
mod_nadp_all

,siteId,model_time,Mod_DOC_mgm2,lat,lon,time,SamplingInt,PREC_tot_mm,PRECC_mm,PRECL_mm
0,AK02,2017-02-23,0.004087,58.900524,225.0,2017-02-22,0.0,0.390721,0.0,0.390721
1,AK02,2017-02-24,0.035680,58.900524,225.0,2017-02-23,0.0,0.522151,0.0,0.522151
2,AK02,2017-02-25,0.000636,58.900524,225.0,2017-02-24,0.0,0.057512,0.0,0.057512
3,AK02,2017-02-26,0.196769,58.900524,225.0,2017-02-25,0.0,5.831723,0.0,5.831723
4,AK02,2017-02-27,0.009516,58.900524,225.0,2017-02-26,0.0,0.227594,0.0,0.227594
...,...,...,...,...,...,...,...,...,...,...
1909,WY08,2018-12-15,0.000002,44.764398,250.0,2018-12-14,11.0,0.054225,0.0,0.054225
1910,WY08,2018-12-16,0.002101,44.764398,250.0,2018-12-15,11.0,0.273937,0.0,0.273937
1911,WY08,2018-12-17,0.000006,44.764398,250.0,2018-12-16,11.0,0.216255,0.0,0.216255
1912,WY08,2018-12-18,0.000001,44.764398,250.0,2018-12-17,11.0,0.023829,0.0,0.023829


In [21]:
#Then group by siteId and sampling int and sum variables below.
mod_nadp_sum = mod_nadp_all.groupby(['siteId', 'SamplingInt', 'lat', 'lon'])[['PREC_tot_mm', 'Mod_DOC_mgm2']].sum().reset_index()
mod_nadp_sum

mod_nadp_sum = pd.merge(mod_nadp_sum, nadp_df, how = 'left', on = ['siteId', 'SamplingInt'])
mod_nadp_sum.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/SamplingInt_Timeseries_NADPsites/PREC_DOC_Timeseries.SamplingInt_Summed_pairedNADPsites.csv')
mod_nadp_sum.head(20)

,siteId,SamplingInt,lat,lon,PREC_tot_mm,Mod_DOC_mgm2,latitude,longitude,dateOn,dateOff,dateOnround,dateOffround,subppt,DOC_mgm2,IntTime
0,AK02,0.0,58.900524,225.0,9.686471,0.335779,58.5139,-134.7843,2017-02-21 18:05:00,2017-02-28 18:28:00,2017-02-22,2017-03-01,8.128,3.16992,7 days 00:23:00
1,AK02,1.0,58.900524,225.0,25.266641,0.437122,58.5139,-134.7843,2017-03-21 18:55:00,2017-03-28 17:50:00,2017-03-22,2017-03-29,12.192,5.97408,6 days 22:55:00
2,AK02,2.0,58.900524,225.0,13.571680,1.109032,58.5139,-134.7843,2017-04-18 18:07:00,2017-04-25 18:01:00,2017-04-19,2017-04-26,8.128,4.79552,6 days 23:54:00
3,AK02,3.0,58.900524,225.0,65.567479,0.803520,58.5139,-134.7843,2017-05-16 16:18:00,2017-05-23 18:23:00,2017-05-17,2017-05-24,39.116,18.77568,7 days 02:05:00
4,AK02,4.0,58.900524,225.0,53.861113,2.006287,58.5139,-134.7843,2017-06-13 18:02:00,2017-06-20 17:17:00,2017-06-14,2017-06-21,78.994,33.96742,6 days 23:15:00
5,AK02,5.0,58.900524,225.0,35.429412,9.456834,58.5139,-134.7843,2017-07-11 17:25:00,2017-07-18 18:11:00,2017-07-12,2017-07-19,77.470,32.53740,7 days 00:46:00
6,AK02,6.0,58.900524,225.0,41.743089,2.394086,58.5139,-134.7843,2017-08-08 17:11:00,2017-08-15 22:30:00,2017-08-09,2017-08-16,47.244,11.33856,7 days 05:19:00
7,AK02,7.0,58.900524,225.0,12.220734,1.016864,58.5139,-134.7843,2017-09-12 18:20:00,2017-09-19 18:50:00,2017-09-13,2017-09-20,11.430,4.34340,7 days 00:30:00
8,AK02,8.0,58.900524,225.0,86.884882,1.427247,58.5139,-134.7843,2017-10-03 19:12:00,2017-10-10 18:50:00,2017-10-04,2017-10-11,53.848,11.30808,6 days 23:38:00
9,AK02,9.0,58.900524,225.0,28.566010,0.221961,58.5139,-134.7843,2017-11-21 19:35:00,2017-11-28 19:41:00,2017-11-22,2017-11-29,30.734,13.52296,7 days 00:06:00
